# LLM-based Usability Evaluation
This notebook guides through the process of the LLM-based Usability Evaluation, the aggregation of issues, and the ranking by severity.

This notebook is divided into four main parts:

1.  **Setup and Configuration**: Loads necessary data, including task descriptions and Nielsen's usability heuristics. Additionally, the connection to the LLM is configured.
2.  **Heuristic Evaluation**: For each user task recording, a multimodal LLM is prompted to identify usability issues based on each of Nielsen's 10 heuristics. The results are saved to a JSON file.
3.  **Issue Aggregation**: The raw evaluation often produces redundant findings across different heuristics. In this step, sentence embeddings are used to find and group similar usability issues, summarizing them into a single, aggregated problem statement and recommendation.
4.  **Severity Ranking**: Finally, an LLM is used to re-rank the aggregated issues based on their potential impact on the user experience, providing a prioritized list of action items.

## Setup and Configuration

In [ ]:
# Define the path to the dataset.
# This path should point to the directory containing the task_description.json, video files, and a 'reports' subdirectory for storing evaluation outputs.

DATA_PATH = "../data/KnowledgeCheckR/"
# DATA_PATH = "../data/EventHelpR/"

In [ ]:
import json
import os

# Load the task description file, which contains details about the application, user personas, and the tasks they are performing.
with open(os.path.join(DATA_PATH, "task_description.json")) as f:
    task_description = json.load(f)

# Load Nielsen's usability heuristics. These will be used one by one to guide the LLM in its evaluation of the user interface.
with open(os.path.join("nielsen_heuristics.json")) as f:
    nielsen_heuristics = json.load(f)

In [ ]:
def load_video(video_path):
    """
    Reads a video file from the given path and returns it as a byte stream.
    This is necessary for passing the video data to the multimodal LLM.
    """
    with open(video_path, "rb") as f:
        video = f.read()
    return video

### Configure LLM
This section defines the core components for interacting with the LLM:

*   **API Key and Model**: Credentials and the specific model to be used.
*   **Response Functions**: Helper functions to send requests to the LLM and parse the JSON response.

In [ ]:
# --- IMPORTANT ---
# Replace with your actual Google AI API key.
GENAI_API_KEY = "YOUR_API_KEY_HERE"

# Define the specific generative model to be used for the evaluation.
GENAI_MODEL = "gemini-2.0-flash-001"

In [ ]:
import re
import json


def repair_json_quotes(json_string: str) -> str:
    """
    Repairs a JSON string by escaping unescaped double quotes within its string values.
    This function iterates through the string character by character to reliably
    determine if a quote is inside a string literal or part of the JSON structure.

    Args:
        json_string: The potentially malformed JSON string.

    Returns:
        A repaired JSON string.
    """
    repaired = []
    in_string = False
    i = 0
    while i < len(json_string):
        char = json_string[i]

        if char == '"':
            # Check if this quote is escaped
            if i > 0 and json_string[i - 1] == '\\':
                # This is an escaped quote, so just append it
                repaired.append(char)
                i += 1
                continue

            # This is an unescaped quote.
            # It can either be a structural quote (toggling in_string)
            # or a quote inside a string that needs to be escaped.
            if not in_string:
                # We are not in a string, so this is an opening structural quote
                in_string = True
                repaired.append(char)
            else:
                # We are inside a string. Is this a closing quote or one to escape?
                # A simple heuristic: if the next non-whitespace character is a comma,
                # closing brace/bracket, or colon, it's likely a structural closing quote.

                # Look ahead for the next non-whitespace character
                next_char_index = i + 1
                while next_char_index < len(json_string) and json_string[next_char_index].isspace():
                    next_char_index += 1

                if next_char_index < len(json_string) and json_string[next_char_index] in [',', '}', ']', ':']:
                    # This looks like a closing structural quote
                    in_string = False
                    repaired.append(char)
                else:
                    # This is a quote inside a string that needs to be escaped
                    repaired.append('\\"')
        else:
            repaired.append(char)

        i += 1

    return "".join(repaired)

In [ ]:
from json import JSONDecodeError
from google import genai
from google.genai.types import Part

def get_response_json(system_prompt, user_prompt, video=None, video_mime_type="video/mp4"):
    """
    Sends a request to the generative AI model and parses the response as JSON.
    This function handles multimodal input (text and video) and is designed for
    structured data extraction from the LLM's response.

    Args:
        system_prompt (str): The system-level instruction for the model.
        user_prompt (str): The user-facing prompt with task details.
        video (bytes, optional): The video data as a byte stream. Defaults to None.
        video_mime_type (str, optional): The MIME type of the video. Defaults to "video/mp4".

    Returns:
        dict or google.generativeai.types.GenerateContentResponse: A dictionary parsed from the JSON
        response. If JSON decoding fails, returns the raw response object.
    """
    client = genai.Client(api_key=GENAI_API_KEY)

    media_content = []
    if video is not None:
        media_content = [Part.from_bytes(data=video, mime_type=video_mime_type)]

    response = client.models.generate_content(
        model=GENAI_MODEL,
        contents=[*media_content, user_prompt],
        config={
            "system_instruction": system_prompt,
            "temperature": 0.0
        }
        )
    try:
        response_json = json.loads(repair_json_quotes(response.candidates[0].content.parts[0].text.lstrip("```json").rstrip("```")))
    except JSONDecodeError:
        print(f"Error decoding JSON. Return the raw response.")
        return response.candidates[0].content.parts[0].text
    return response_json

def get_response_text(user_prompt):
    """
    Sends a text-only request to the generative AI model and returns the response as a string.
    This function is used for simpler, text-based summarization tasks.

    Args:
        user_prompt (str): The user-facing prompt.

    Returns:
        str: The text content of the model's response.
    """
    client = genai.Client(api_key=GENAI_API_KEY)
    response = client.models.generate_content(
        model=GENAI_MODEL,
        contents=[user_prompt],
        config={
            "temperature": 0.0
        }
    )
    return response.candidates[0].content.parts[0].text

## Heuristic Evaluation
This section executes the core logic of the evaluation.

The `run_usability_evaluation` function iterates through each persona and their assigned tasks. For each task, it loads the corresponding screen recording and then loops through all 10 of Nielsen's heuristics.

For each heuristic, it:
1.  Constructs a detailed prompt using the context (persona, task, app description) and the specific heuristic.
2.  Sends the prompt and the video to the multimodal LLM.
3.  Appends the structured JSON response to a list.

Finally, the collected evaluations are saved to a timestamped JSON file in the `reports` directory.

In [ ]:
# The system prompt sets the overall context and persona for the LLM.
# It instructs the model to act as a usability expert and follow a specific evaluation methodology.
EVALUATION_SYSTEM_PROMPT = f"""
You are a highly experienced usability expert.
Your task is to evaluate the software interface using Nielsen’s usability heuristics.
Use the provided user interaction recording and contextual descriptions to support a detailed, heuristic-based evaluation.
Assess each heuristic independently.
Base your evaluations strictly on the provided materials and context — do not make assumptions.
Provide structured findings with clearly described issues and actionable recommendations.
"""

In [ ]:
def get_evaluation_user_prompt(heuristic_name, heuristic_explanation, app_description, persona, task):
    """
    Constructs the detailed user prompt for the LLM-based usability evaluation.

    Args:
        heuristic_name (str): The name of the Nielsen heuristic to be evaluated.
        heuristic_explanation (str): A brief explanation of the heuristic.
        app_description (str): A description of the application under evaluation.
        persona (str): A description of the user persona performing the task.
        task (str): A description of the task the user is attempting to complete.

    Returns:
        str: The fully formatted prompt to be sent to the LLM.
    """

    prompt = ""
    prompt += f"App Description:\n{app_description}\n"
    prompt += f"Persona:\n{persona}\n"
    prompt += f"User Task:\n{task}\n"

    prompt += f"""
Heuristic Under Evaluation:
{heuristic_name}

Heuristic Explanation:
{heuristic_explanation}

Evaluation Instructions:
Evaluate the interface for the heuristic listed above. Follow these steps:
1. Identify any usability issues related to the heuristic and explain each issue.
2. Provide a practical, actionable recommendation to fix each issue.
3. Assign a severity level using:
   - No Issue
   - Minor Issue
   - Major Issue

Response Guideline:
Keep each issue’s analysis and recommendation concise (1–5 sentences). Avoid restating the heuristic definition.

Format your response as JSON as follows:
{{
  "heuristic": "{heuristic_name}",
  "issues": [
    {{
      "description": "[Brief description of the usability issue]",
      "recommendation": "[Actionable suggestion for improvement]",
      "severity": "[No Issue | Minor Issue | Major Issue]"
    }}
    // Repeat for each issue
  ]
}}

If no issues are found:
{{
  "heuristic": "{heuristic_name}",
  "issues": []
}}
Answer with JSON only.
"""

    return prompt

In [ ]:
from datetime import datetime

def store_usability_evaluation(evaluation, model_name):
    """
    Saves the raw usability evaluation report to a JSON file.
    The file is named with the model name and a timestamp to ensure uniqueness.

    Args:
        evaluation (list): A list of dictionaries, where each dictionary is the
                           JSON response from the LLM for a single heuristic evaluation.
        model_name (str): The name of the LLM used for the evaluation.
    """
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    with open(os.path.join(DATA_PATH, "reports", f"evaluation_{model_name}_{timestamp}.json"), "w") as f:
        json.dump(evaluation, f, indent=2)

In [ ]:
def run_usability_evaluation():
    """
    Orchestrates the usability evaluation process.
    It iterates through each persona and task defined in the task_description.json file.
    For each task, it performs a heuristic evaluation using the associated video recording.

    Returns:
        list: A list containing the JSON responses from the LLM for all evaluated
              persona-task-heuristic combinations.
    """
    video_task_evaluation = []

    for persona in task_description["personas"]:
        for task in persona["tasks"]:
            encoded_video = load_video(os.path.join(DATA_PATH, task["video_file"]))
            for heuristic_id, heuristic in nielsen_heuristics.items():
                id = f"{task_description['app_id']}_{persona['persona_id']}_{task['task_id']}_{heuristic_id}"
                print(f"Processing {id}")
                user_prompt = get_evaluation_user_prompt(heuristic["name"], heuristic["explanation"], task_description["description"],
                                                         persona["description"], task["description"])

                response_json = get_response_json(EVALUATION_SYSTEM_PROMPT, user_prompt, video=encoded_video, video_mime_type="video/mp4")

                response_json["id"] = id
                video_task_evaluation.append(response_json)

    return video_task_evaluation


In [ ]:
# This cell triggers the entire evaluation process.
# Note: This can take a longer time to run, as it makes multiple calls to the LLM API.

print("Starting LLM-based usability evaluation...")
video_task_evaluation = run_usability_evaluation()

# After the evaluation is complete, store the raw results in a JSON file.
store_usability_evaluation(video_task_evaluation, GENAI_MODEL)
print("Evaluation complete and report stored.")

## Issue Aggregation
A single user task can reveal the same underlying usability problem for different heuristics. For example, a confusing icon might be flagged as a violation of "Recognition rather than recall" (H6) and also "Aesthetic and minimalist design" (H8).

This section aims to de-duplicate and consolidate these overlapping findings. The process is as follows:

1.  **Group by Task**: First, all identified issues are grouped by the user task they belong to (e.g., all issues from `EHR_P1_T1`).
2.  **Semantic Clustering**: Within each task group, a sentence-embedding model (`all-MiniLM-L6-v2`) is used to convert issue descriptions into numerical vectors. Afterward, the cosine similarity between these vectors is computed.
3.  **Create Clusters**: Issues with a similarity score above a certain threshold (e.g., 0.70) are grouped into a single "cluster."
4.  **Summarize and Finalize**: For each cluster, an LLM is used to:
    *   Synthesize the various issue descriptions into one concise problem statement.
    *   Combine the recommendations into a single, actionable suggestion.
    *   Determine the overall severity via a majority vote.
5.  **Store Report**: The final, aggregated report is saved to a new JSON file.

In [ ]:
def load_evaluation(evaluation_path):
    """
    Loads a raw evaluation JSON file from the specified path.

    Args:
        evaluation_path (str): The full path to the evaluation file.

    Returns:
        list: The loaded evaluation data.
    """

    with open(evaluation_path) as f:
        evaluation = json.load(f)
    return evaluation

def store_aggregated_report(report, name):
    """
    Saves the aggregated and summarized report to a JSON file.

    Args:
        report (dict): The finalized report dictionary.
        name (str): The name to use for the output file.
    """
    with open(os.path.join(DATA_PATH, "reports", f"report_{name}"), "w") as f:
        json.dump(report, f, indent=2)

In [ ]:
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict
import re

# Initialize the sentence-embedding model.
model = SentenceTransformer("all-MiniLM-L6-v2")

def extract_task_id(full_id):
    """
    Extracts the task-level portion of an ID string.
    Example: 'EHR_P1_T1_H4' -> 'EHR_P1_T1'
    """
    match = re.match(r'^[A-Z]+_P\d+_T\d+', full_id)
    return match.group(0) if match else full_id

def aggregate_issues_by_task_and_similarity(data, threshold=0.7):
    """
    Groups raw usability issues first by task and then by semantic similarity.

    Args:
        data (list): The raw evaluation data from the LLM.
        threshold (float): The cosine similarity score required to group
                           two issues into the same cluster.

    Returns:
        tuple: A tuple containing:
            - aggregated_results (dict): A dictionary where keys are task IDs and
                                         values are lists of issue clusters.
            - similarities (list): A list of all computed similarity scores, useful for debugging.
    """
    task_groups = defaultdict(list)

    # Step 1: Group all identified issues by their parent task ID (e.g., KCR_P1_T1).
    for entry in data:
        task_id = extract_task_id(entry["id"])
        for issue in entry["issues"]:
            task_groups[task_id].append({
                "heuristic": entry["heuristic"],
                "description": issue["description"],
                "recommendation": issue["recommendation"],
                "severity": issue["severity"]
            })

    similarities = []
    aggregated_results = {}

    # Step 2: For each task, cluster issues based on semantic similarity.
    for task_id, issues in task_groups.items():
        descriptions = [i["description"] for i in issues]
        embeddings = model.encode(descriptions, convert_to_tensor=True)

        visited = set()
        clusters = []

        for i in range(len(issues)):
            if i in visited:
                continue

            # Start a new cluster with the current issue
            cluster = {
                "descriptions": [issues[i]["description"]],
                "recommendations": [issues[i]["recommendation"]],
                "severities": [issues[i]["severity"]],
                "heuristics": [issues[i]["heuristic"]]
            }
            visited.add(i)

            # Compare with all other issues in the same task group.
            for j in range(i + 1, len(issues)):
                if j in visited:
                    continue

                # Calculate cosine similarity between issue descriptions.
                similarity_score = util.cos_sim(embeddings[i], embeddings[j]).item()
                similarities.append(similarity_score)

                if similarity_score >= threshold:
                    # If similar, add the issue to the current cluster.
                    cluster["descriptions"].append(issues[j]["description"])
                    cluster["recommendations"].append(issues[j]["recommendation"])
                    cluster["severities"].append(issues[j]["severity"])
                    cluster["heuristics"].append(issues[j]["heuristic"])
                    visited.add(j)
            clusters.append(cluster)

        aggregated_results[task_id] = clusters

    return aggregated_results, similarities

In [ ]:
from collections import Counter

def majority_severity_vote(severities):
    """
    Determines the final severity for a cluster of issues.
    It uses a majority vote. If there's a tie, 'Major Issue' is prioritized.

    Args:
        severities (list): A list of severity strings (e.g., ['Minor Issue', 'Major Issue']).

    Returns:
        str: The winning severity string.
    """
    counts = Counter(severities)
    # If there's a tie between the top two, check if one is 'Major Issue'.
    most_common = counts.most_common()
    if len(most_common) > 1 and most_common[0][1] == most_common[1][1]:
        return "Major Issue" if "Major Issue" in [s[0] for s in most_common] else most_common[0][0]
    return most_common[0][0]

In [ ]:
def summarize_with_llm(texts, prompt_type="description"):
    """
    Uses an LLM to summarize a list of issue descriptions or recommendations.

    Args:
        texts (list): A list of strings to be summarized.
        prompt_type (str): Either "description" or "recommendation" to use the correct prompt.

    Returns:
        str: The summarized text.
    """
    joined_texts = "\n".join(texts)

    if prompt_type == "description":
        prompt = f"""
Summarize the following usability issue descriptions into one clear, concise problem statement (1-5 sentences) that captures the main user pain point.
Avoid repeating examples.
Focus on the core usability problem.

Descriptions:
{joined_texts}

Answer directly with the summary.
"""
    elif prompt_type == "recommendation":
        prompt = f"""
Summarize the following usability improvement recommendations into one actionable suggestion that resolves the identified issues.
Combine overlapping suggestions and avoid redundancy.

Recommendations:
{joined_texts}

Answer directly with the summary.
"""

    summary = get_response_text(prompt)
    return summary

In [ ]:
def finalize_aggregated_report(aggregated_data):
    """
    Summarizes each cluster of issues and determines the final severity.

    Args:
        aggregated_data (dict): The output from `aggregate_issues_by_task_and_similarity`.

    Returns:
        dict: The final, summarized report, ready for review.
    """
    final_output = {}

    for task_id, clusters in aggregated_data.items():
        final_output[task_id] = []
        print(f"--- Finalizing report for task: {task_id} ---")

        for i, cluster in enumerate(clusters):
            print(f"Summarizing cluster {i+1}/{len(clusters)}...")

            # If a cluster only has one issue, no need to summarize.
            desc = summarize_with_llm(cluster["descriptions"], "description") if len(cluster["descriptions"]) > 1 else cluster["descriptions"][0]
            rec = summarize_with_llm(cluster["recommendations"], "recommendation") if len(cluster["recommendations"]) > 1 else cluster["recommendations"][0]

            summary = {
                "description": desc,
                "recommendation": rec,
                "severity": majority_severity_vote(cluster["severities"]),
                "heuristics": sorted(list(set(cluster["heuristics"])))  # Get unique, sorted list
            }
            final_output[task_id].append(summary)

    return final_output


In [ ]:
# --- Aggregation Execution ---

# Specify the name of the raw evaluation file to be processed.
# This should be one of the files generated in the previous step.
EVALUATION_FILE = "evaluation_gemini-2.0-flash-001.json"

# Load the raw report
print(f"Loading raw evaluation file: {EVALUATION_FILE}")
evaluation_report = load_evaluation(os.path.join(DATA_PATH, "reports", EVALUATION_FILE))

# Run the aggregation and clustering process.
# The threshold can be adjusted. A higher value means descriptions must be more similar to be grouped.
print("Aggregating issues by task and similarity...")
aggregated_report, similarities = aggregate_issues_by_task_and_similarity(evaluation_report, threshold=0.70)

# Summarize the clusters and finalize the report.
# This step involves making more calls to the LLM API for summarization.
print("Finalizing aggregated report with LLM summarization...")
finalized_report = finalize_aggregated_report(aggregated_report)

# Store the final, aggregated report.
store_aggregated_report(finalized_report, EVALUATION_FILE.split("/")[-1].lstrip("evaluation_"))

print("Aggregation complete. Final report stored.")

### Severity Ranking
While the aggregation step assigns a severity to each issue, it's based on a simple majority vote from potentially conflicting initial assessments. This final section provides a more holistic ranking.

Here's the process:

1.  **Load Aggregated Report**: Load the summarized report generated in the previous section.
2.  **Prepare for Ranking**: For each task, a clean list of the summarized issues is prepared by removing the preliminary severity and heuristic data to avoid biasing the ranking model.
3.  **Construct Ranking Prompt**: A new system prompt instructs the LLM to act as a usability expert and re-rank the provided list of issues. The prompt asks the model to consider factors like task success, user frustration, and frequency.
4.  **Execute Ranking**: The LLM receives the list of issues and returns a re-ranked JSON array, with a justification for each issue's position in the ranking.
5.  **Store Final Ranked Report**: The final, ranked report is saved to a new JSON file, providing a clear, prioritized list of usability problems to address.

In [ ]:
def load_report(report_path):
    """
    Loads a summarized (aggregated) report JSON file.

    Args:
        report_path (str): The path to the report file, relative to the 'reports' directory.

    Returns:
        dict: The loaded report data.
    """
    with open(os.path.join(DATA_PATH, "reports", report_path)) as f:
        report = json.load(f)
    return report

def store_ranking_report(report, name):
    """
    Saves the final, ranked report to a JSON file.

    Args:
        report (dict): The ranked report data.
        name (str): The name to use for the output file.
    """
    with open(os.path.join(DATA_PATH, "reports", f"ranking_report_{name}"), "w") as f:
        json.dump(report, f, indent=2)

In [ ]:
# This system prompt is specifically designed for the ranking task.
# It instructs the LLM to re-evaluate the severity of a list of known issues based on a more holistic set of criteria than the simple heuristic evaluation.
RANKING_SYSTEM_PROMPT = f"""
You are a highly experienced usability expert applying Nielsen’s heuristics.
Your task is to re-rank identified usability issues by actual severity.
Consider the impact on task success and user frustration, frequency and likelihood of occurrence, and effort required to diagnose and recover.
Base your evaluations strictly on the provided materials and context — do not make assumptions.
Provide a ranked list from most to least severe.
"""

In [ ]:
def get_ranking_user_prompt(report):
    """
    Constructs the user prompt for the issue ranking task.

    Args:
        report (list): A list of summarized issues to be ranked.

    Returns:
        str: The fully formatted prompt for the ranking task.
    """
    prompt = f"""
Evaluate and re-rank the following usability issues by severity.
Each issue includes a description and a recommendation on how to fix the issue.

Format your response as JSON as follows:
[
    {{
      "id": "[Issue identifier]",
      "justification": "[Explanation of the issue severity.]",
    }}
    // Repeat for each issue
  ]
]

Answer with JSON only.

Usability issues to rank:
{report}
"""

    return prompt

In [ ]:
import copy

# --- Ranking Execution ---

# Specify the aggregated report file you want to rank.
# This file should be the output from the "Issue Aggregation" step.
REPORT_FILE = "report_gemini-2.0-flash-001.json"

print(f"Loading aggregated report: {REPORT_FILE}")
report = load_report(REPORT_FILE)

# This dictionary will hold the final, ranked issues, grouped by task ID.
ranked_report = {}

# Iterate through each task group (e.g., 'EHR_P1_T1') in the report.
for task_id, issues in report.items():
    print(f"--- Ranking issues for task: {task_id} ---")

    # Create a clean copy of the issues to send to the LLM for ranking.
    # Assign a simple, temporary ID (e.g., "I_0", "I_1") to each issue.
    # This allows the LLM to reference them easily in its ranked response.
    issues_to_rank = []
    for i, issue in enumerate(issues):
        # We remove preliminary severity and heuristic data to avoid biasing the ranking model.
        issue_for_llm = {
            "id": f"I_{i}",
            "description": issue["description"],
            "recommendation": issue["recommendation"]
        }
        issues_to_rank.append(issue_for_llm)

        # Also add the temporary ID back to our original list of issues.
        # This allows us to map the LLM's ranked response back to the full issue data.
        issues[i]["id"] = f"I_{i}"

    # Construct the prompt and get the ranked list of issues from the LLM.
    user_prompt = get_ranking_user_prompt(issues_to_rank)
    response_json = get_response_json(RANKING_SYSTEM_PROMPT, user_prompt)

    # Reconstruct the final list of issues in the order provided by the LLM.
    ranked_issues = []
    if response_json and isinstance(response_json, list):
        for ranked_item in response_json:
            # Find the original issue data using the temporary ID from the LLM's response.
            original_issue = next((i for i in issues if i.get("id") == ranked_item.get("id")), None)

            if original_issue:
                # Create a clean copy to avoid modifying the original list in place.
                final_issue = original_issue.copy()
                # Add the LLM's justification for the ranking.
                final_issue["rank_justification"] = ranked_item.get("justification", "No justification provided.")
                # Remove the temporary ID.
                del final_issue["id"]
                ranked_issues.append(final_issue)

    ranked_report[task_id] = ranked_issues

# Store the final, ranked report in a new JSON file.
store_ranking_report(ranked_report, REPORT_FILE.lstrip("report_"))
print("Ranking complete. Final ranked report stored.")